# Import Libraries

In [1]:
%load_ext autoreload
%autoreload 2
import argparse
import os
import yaml
import copy
import torch
import random
import numpy as np

from utils import dict2namespace, get_runner, namespace2dict
import torch.multiprocessing as mp
import torch.distributed as dist

import sys

from runners.DiffusionBasedModelRunners import BBDMRunner
# from model.VQGAN.taming.data.custom import CustomTest, CustomTestClariGAN
from datasets.custom import CustomAlignedDataset
from runners.utils import weights_init, get_optimizer, get_dataset, make_dir, get_image_grid, save_single_image
from torch.utils.data import DataLoader

from tqdm import tqdm
import matplotlib.pyplot as plt
from PIL import Image

c:\Users\ammic\Desktop\ClariGAN-DL\BBDM\model\BrownianBridge\BrownianBridgeModel.py:7: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


# Functions

In [2]:
def parse_args_and_config():
    parser = argparse.ArgumentParser(description=globals()['__doc__'])

    parser.add_argument('-c', '--config', type=str, default='BB_base.yml', help='Path to the config file')
    parser.add_argument('-s', '--seed', type=int, default=1234, help='Random seed')
    parser.add_argument('-r', '--result_path', type=str, default='results', help="The directory to save results")

    parser.add_argument('-t', '--train', action='store_true', default=False, help='train the model')
    parser.add_argument('--sample_to_eval', action='store_true', default=False, help='sample for evaluation')
    parser.add_argument('--sample_at_start', action='store_true', default=False, help='sample at start(for debug)')
    parser.add_argument('--save_top', action='store_true', default=False, help="save top loss checkpoint")

    parser.add_argument('--gpu_ids', type=str, default='0', help='gpu ids, 0,1,2,3 cpu=-1')
    parser.add_argument('--port', type=str, default='12355', help='DDP master port')

    parser.add_argument('--resume_model', type=str, default=None, help='model checkpoint')
    parser.add_argument('--resume_optim', type=str, default=None, help='optimizer checkpoint')

    parser.add_argument('--max_epoch', type=int, default=None, help='optimizer checkpoint')
    parser.add_argument('--max_steps', type=int, default=None, help='optimizer checkpoint')

    args = parser.parse_args()

    with open(args.config, 'r') as f:
        dict_config = yaml.load(f, Loader=yaml.FullLoader)

    namespace_config = dict2namespace(dict_config)
    namespace_config.args = args

    if args.resume_model is not None:
        namespace_config.model.model_load_path = args.resume_model
    if args.resume_optim is not None:
        namespace_config.model.optim_sche_load_path = args.resume_optim
    if args.max_epoch is not None:
        namespace_config.training.n_epochs = args.max_epoch
    if args.max_steps is not None:
        namespace_config.training.n_steps = args.max_steps

    dict_config = namespace2dict(namespace_config)

    return namespace_config, dict_config


def set_random_seed(SEED=1234):
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.enabled = True
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def DDP_run_fn(rank, world_size, config):
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = config.args.port
    dist.init_process_group(backend='nccl', rank=rank, world_size=world_size)

    set_random_seed(config.args.seed)

    local_rank = dist.get_rank()
    torch.cuda.set_device(local_rank)
    config.training.device = [torch.device("cuda:%d" % local_rank)]
    print('using device:', config.training.device)
    config.training.local_rank = local_rank
    return # STOP FROM DEFINING RUNNER INSTANCE

    runner = get_runner(config.runner, config)
    if config.args.train:
        runner.train()
    else:
        with torch.no_grad():
            runner.test()
    return


def CPU_singleGPU_launcher(config):
    set_random_seed(config.args.seed)
    return # STOP FROM DEFINING RUNNER INSTANCE
    runner = get_runner(config.runner, config)
    if config.args.train:
        runner.train()
    else:
        with torch.no_grad():
            runner.test()
    return


def DDP_launcher(world_size, run_fn, config):
    raise Exception("Not Allowing Multiple GPU Inference")
    mp.spawn(run_fn,
             args=(world_size, copy.deepcopy(config)),
             nprocs=world_size,
             join=True)


def main():
    nconfig, dconfig = parse_args_and_config()
    args = nconfig.args

    gpu_ids = args.gpu_ids
    if gpu_ids == "-1": # Use CPU
        nconfig.training.use_DDP = False
        nconfig.training.device = [torch.device("cpu")]
        CPU_singleGPU_launcher(nconfig)
    else:
        gpu_list = gpu_ids.split(",")
        if len(gpu_list) > 1:
            os.environ['CUDA_VISIBLE_DEVICES'] = gpu_ids
            nconfig.training.use_DDP = True
            DDP_launcher(world_size=len(gpu_list), run_fn=DDP_run_fn, config=nconfig)
        else:
            nconfig.training.use_DDP = False
            nconfig.training.device = [torch.device(f"cuda:{gpu_list[0]}")]
            CPU_singleGPU_launcher(nconfig)
    return nconfig


# Define Model From Config and Load Checkpoint

In [3]:
sys.argv = [
    "test_set_results.ipynb",  # Placeholder for script name
    "--config", r"C:\Users\ammic\Desktop\ClariGAN-DL\BBDM\configs\Template-LBBDM-f16_imagenetVQGAN_BF.yaml",
    "--gpu_ids", "-1",
    "--resume_model", r"C:\Users\ammic\Desktop\ClariGAN-DL\BBDM\results\BF_imagenetVQGAN_finetuned\LBBDM-f16\checkpoint\top_model_epoch_40.pth", # r"C:\Users\ammic\Desktop\ClariGAN-DL\BBDM\results\ClariGAN_imagenetVQGAN_v1\LBBDM-f16\checkpoint\top_model_epoch_56.pth",
    "--resume_optim", r"C:\Users\ammic\Desktop\ClariGAN-DL\BBDM\results\BF_imagenetVQGAN_finetuned\LBBDM-f16\checkpoint\top_optim_sche_epoch_40.pth" # r"C:\Users\ammic\Desktop\ClariGAN-DL\BBDM\results\ClariGAN_imagenetVQGAN_v1\LBBDM-f16\checkpoint\top_optim_sche_epoch_56.pth"
]

In [4]:
if __name__ == "__main__":
    nconfig = main()

In [5]:
nconfig.data.test.batch_size = 1

In [6]:
nconfig.data.dataset_config.dataset_path = r"C:\Users\ammic\Downloads\BFDF-dataset"

In [7]:
runner = get_runner(nconfig.runner, nconfig)

save training results to results\BF_imagenetVQGAN_finetuned\LBBDM-f16\
Working with z of shape (1, 256, 16, 16) = 65536 dimensions.


c:\Users\ammic\Desktop\ClariGAN-DL\BBDM\model\VQGAN\vqgan.py:64: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  sd = torch.load(path, map_location="cpu")["state_dict"]


Restored from C:\Users\ammic\Desktop\ClariGAN-DL\taming-transformers\logs\2025-02-19T21-05-24_BF_finetune\checkpoints\epoch=000026.ckpt
load vqgan from C:\Users\ammic\Desktop\ClariGAN-DL\taming-transformers\logs\2025-02-19T21-05-24_BF_finetune\checkpoints\epoch=000026.ckpt
get parameters to optimize: UNet
Total Number of parameter: 334.75M
Trainable Number of parameter: 258.68M
load model LBBDM-f16 from C:\Users\ammic\Desktop\ClariGAN-DL\BBDM\results\BF_imagenetVQGAN_finetuned\LBBDM-f16\checkpoint\top_model_epoch_40.pth


c:\Users\ammic\anaconda3\envs\BBDM\lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(
c:\Users\ammic\Desktop\ClariGAN-DL\BBDM\runners\BaseRunner.py:115: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use

In [8]:
bbdmnet = runner.initialize_model(nconfig)

Working with z of shape (1, 256, 16, 16) = 65536 dimensions.
Restored from C:\Users\ammic\Desktop\ClariGAN-DL\taming-transformers\logs\2025-02-19T21-05-24_BF_finetune\checkpoints\epoch=000026.ckpt
load vqgan from C:\Users\ammic\Desktop\ClariGAN-DL\taming-transformers\logs\2025-02-19T21-05-24_BF_finetune\checkpoints\epoch=000026.ckpt


In [9]:
ckpt_path = r"C:\Users\ammic\Desktop\ClariGAN-DL\BBDM\results\BF_imagenetVQGAN_finetuned\LBBDM-f16\checkpoint\top_model_epoch_40.pth"
bbdmnet.load_state_dict(torch.load(ckpt_path, weights_only=True, map_location='cpu')['model']) # nconfig.training.device[0]

<All keys matched successfully>

# Define Train/Val/Test Set

In [10]:
train_dataset, val_dataset, test_dataset = get_dataset(nconfig.data)

train_loader = DataLoader(train_dataset,
                            batch_size=nconfig.data.train.batch_size,
                            shuffle=nconfig.data.train.shuffle,
                            num_workers=8,
                            drop_last=True)

val_loader = DataLoader(val_dataset,
                        batch_size=nconfig.data.val.batch_size,
                        shuffle=nconfig.data.val.shuffle,
                        num_workers=8,
                        drop_last=True)

test_loader = DataLoader(test_dataset,
                            batch_size=nconfig.data.test.batch_size,
                            shuffle=False,
                            num_workers=8,
                            drop_last=True)

# Train the Adapter

In [11]:
import torch
from AdapterNetwork import *

In [12]:
# Check if a GPU is available and set the device accordingly
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Instantiate the AdapterNetwork object
input_channels = 3  # For RGB images
output_channels = 3  # Assuming we want RGB output


# Create an instance of the AdapterNetwork
adapter = AdapterNetwork(
    input_channels=input_channels,
    output_channels=output_channels,
)

# Load the model onto the specified device (GPU/CPU)
adapter.to(device)

# Optionally, print the model summary or check if it's on the correct device
# print(adapter)
print(f"Model loaded to device: {device}")

Loaded pretrained weights for efficientnet-b0
Model loaded to device: cuda


In [13]:
from torchvision.utils import make_grid, save_image

@torch.no_grad()
def get_image_grid(batch, grid_size=4, to_normal=True):
    batch = batch.detach().clone()
    image_grid = make_grid(batch, nrow=grid_size)
    if to_normal:
        image_grid = image_grid.mul_(0.5).add_(0.5).clamp_(0, 1.)
    image_grid = image_grid.mul_(255).add_(0.5).clamp_(0, 255).permute(1, 2, 0).to('cpu', torch.uint8).numpy()
    return image_grid

In [14]:
# Define the loss function and optimizer
criterion = torch.nn.MSELoss()  # Use CrossEntropyLoss for classification tasks
optimizer = torch.optim.Adam(adapter.parameters(), lr=0.001)

num_epochs = 10  # Define the number of epochs
bbdmnet.cpu()
bbdmnet.eval()
for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")

    # Use tqdm for progress bar
    pbar = tqdm(test_loader, total=len(train_loader), smoothing=0.01)
    batch_size = runner.config.data.test.batch_size
    sample_num = runner.config.testing.sample_num
    grid_size = batch_size  # Number of samples per batch

    # Iterate over the test batches
    for test_batch in pbar:
        torch.cuda.empty_cache()
        (x, _), (x_cond, _) = test_batch

        # Move data to device
        x = x.to(device)
        x_cond = x_cond.to(device)

        with torch.no_grad():
            bbdm_outputs = []
            bbdmnet.cuda()

            # Generate samples using the `bbdmnet` model
            for j in range(sample_num):
                # Assuming `bbdmnet.sample` returns outputs of shape (B, C, H, W)
                result = bbdmnet.sample(x_cond, clip_denoised=runner.config.testing.clip_denoised)

                # Move to CPU and append to the list
                bbdm_outputs.append(result.detach().cpu())

                del result

            # Convert the list of outputs into a tensor
            adapter_input = torch.stack(bbdm_outputs)  # Shape becomes (sample_num, B, C, H, W)

            # Free GPU memory used by bbdmnet
            del bbdm_outputs
            bbdmnet.cpu()
            torch.cuda.empty_cache()


        # Now, pass the output into the Adapter Network
        adapter_input = adapter_input.to(device)
        adapter_output = adapter(adapter_input, x_cond)

        del adapter_input  # Free memory


        # Compute the loss
        loss = criterion(adapter_output, x)  # Assuming `x_cond` is the target ground truth

        del x_cond, x, adapter_output # Free memory

        # Zero the gradients
        optimizer.zero_grad()

        # Backpropagate and optimize
        loss.backward()
        optimizer.step()

        # Update progress bar
        pbar.set_description(f"Loss: {loss.item():.4f}")
        del loss  # Free memory

    # # Optionally, save the model after every epoch
    # torch.save(adapter.state_dict(), f"adapter_model_epoch_{epoch+1}.pth")
    # print(f"Epoch {epoch+1} complete. Model saved.")

    # Optionally, print the loss at the end of the epoch
    # print(f"Epoch {epoch+1} completed with loss: {loss.item():.4f}")


Epoch 1/10


Loss: 0.1427:   0%|          | 1/542 [02:53<26:07:56, 173.89s/it]


OutOfMemoryError: CUDA out of memory. Tried to allocate 5.00 GiB. GPU 0 has a total capacity of 11.99 GiB of which 0 bytes is free. Of the allocated memory 26.18 GiB is allocated by PyTorch, and 65.92 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
"""
pbar = tqdm(test_loader, total=len(train_loader), smoothing=0.01)
batch_size = runner.config.data.test.batch_size
sample_num = runner.config.testing.sample_num
grid_size = batch_size
for test_batch in pbar:
    (x, x_name), (x_cond, x_cond_name) = test_batch
    x = x.to(runner.config.training.device[0])
    x_cond = x_cond.to(runner.config.training.device[0])

    outputs = []
    for j in range(sample_num):
        result = bbdmnet.sample(x_cond, clip_denoised=runner.config.testing.clip_denoised).to('cpu')
        outputs.append(result.detach())
        # plt.imshow(x_cond.squeeze(0).permute(1,2,0).cpu())
        # plt.show()
        # image_grid = get_image_grid(outputs[:grid_size], grid_size=grid_size, to_normal=True)
        # plt.imshow(image_grid)
        # plt.show()
    outputs = torch.from_numpy(np.array(outputs)).squeeze(1)
    print(outputs.shape)

    # feed input to adapter

"""
